# Concept 3: Handling a Broken Link

Feeds a nonsense review through the summarize chain, then validates the result with a RunnableLambda before it reaches classify.

## Setup
Store your key in Colab **Secrets** (key icon, left sidebar) as `OPENAI_API_KEY`.

In [ ]:
!pip install -q langchain langchain-openai langchain-core

In [ ]:
from google.colab import userdata
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-4o-mini", api_key=userdata.get("OPENAI_API_KEY"))

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda

In [ ]:
summarize_prompt = ChatPromptTemplate.from_template("Summarize this review in one sentence:\n{review}")
classify_prompt = ChatPromptTemplate.from_template("Classify this as positive or negative. Respond with only one word:\n{summary}")
summarize_chain = summarize_prompt | llm | StrOutputParser()
classify_chain = classify_prompt | llm | StrOutputParser()

In [ ]:
def validate(summary):
    if len(summary.split()) < 3:
        raise ValueError(f"Broken link: summary too short ({summary!r})")
    return {"summary": summary}
full_chain = summarize_chain | RunnableLambda(validate) | classify_chain

In [ ]:
broken_review = "asdkfj laksjdf qpoiuweiur"
try:
    print(full_chain.invoke({"review": broken_review}))
except ValueError as e:
    print(e)

## How to Extend
Try an empty string as the review instead of gibberish. Tighten validate() to also catch apology-style responses like 'I'm sorry' or 'cannot'.